# Asset Subcategory ETL & Data Cleaning
### AMS Legacy Migration to Smart Microsoft SQL Server (`subcategories` table)

---

## 1. Overview & Business Context
This notebook performs data cleaning, normalization, relational mapping, and deduplication for the **Subcategories** dataset from legacy AMS (`data/ams/tb_assets_category_202609291319.csv`).

The goal is to prepare clean, verified data ready for direct import into the Microsoft SQL Server **`subcategories`** table for the new **Smart** application, linked to their parent categories established in `notebooks/category.ipynb`.

### Target Microsoft SQL Server Schema (`subcategories`)
| Column Name | SQL Type | Constraints | Description |
|:---|:---|:---|:---|
| `id` | `BIGINT` | `IDENTITY(1,1) PRIMARY KEY` | Sequential primary key (`1, 2, ..., N`) |
| `code` | `VARCHAR(9)` | `UNIQUE NOT NULL` | Standardized `<PARENT_CODE>-<CHILD_CODE>` format (e.g. `FUR-KK`, `PDGR-ACS`) |
| `name` | `NVARCHAR(255)` | `NOT NULL` | Standardized subcategory name in Title Case |
| `description` | `NVARCHAR(MAX)` | `NULL` | Subcategory description from legacy AMS |
| `is_consumable` | `BIT` | `NOT NULL DEFAULT 0` | Set to `0` (`False`) for all fixed asset/equipment subcategories |
| `category_id` | `BIGINT` | `NOT NULL FOREIGN KEY` | Foreign key referencing `categories.id` (1 to 12) |
| `created_at` | `DATETIME` | `NOT NULL` | Record creation timestamp (`YYYY-MM-DD HH:MM:SS`) |
| `updated_at` | `DATETIME` | `NOT NULL` | Record update timestamp (`YYYY-MM-DD HH:MM:SS`) |

---

## 2. Key Data Cleaning & Business Rules
1. **Parent-Child Code Concatenation**:
   - Follows the pattern established in Smart application seeder (`MasterSeeder.php`): `<PARENT_CODE>-<CHILD_CODE>` (e.g., `FUR-KK`, `COMP-NB`, `MON-LCD`).
   - Every combined code is strictly $\le 9$ characters, complying with `VARCHAR(9)`.
2. **Reclassification of Water Heater to Mesin (`MES`)**:
   - In AMS under `ACC` (`110000`), both `WHITEBOARD` and `WATER HEATER` had numbering code `WH`.
   - `Water Heater` (`110022`) is mechanical/electrical building equipment and is reclassified under **`Mesin` (`MES`)** as **`MES-WTH`**.
   - `Whiteboard` (`110002`) remains under `Aksesoris` (`ACC`) with standardized code **`ACC-WB`**.
3. **Consolidation of Duplicate Items**:
   - **`LIFT`**: In AMS, Lift was created under both Mesin (`180005`, `LIF`) and Elektronik (`240017`, `LT`). Merged into **`MES-LIF`** under Mesin. Both legacy codes (`180005` and `240017`) map to `MES-LIF`.
   - **`MOBIL`**: In AMS, `200001` (`KENDARAAN`, desc: *Operasional RE*) and `200002` (`MOBIL`) are consolidated into **`KEND-MO`** (`Mobil`), reserving room for future vehicle separation (e.g., motor). Both legacy codes (`200001` and `200002`) map to `KEND-MO`.
4. **Relegated Subcategories Parent Linking**:
   - The 24 subcategories that had `parent = NULL` in AMS (e.g. `AC Split`, `AC Portable`, `GENSET`, `CCTV`, `TV`) are automatically mapped to their parent categories (`PDGR`, `MES`, `ELEK`) based on their code prefix (`26xxxx`, `18xxxx`, `24xxxx`).
   - Spelling corrected for `AC FORTABLE` $\rightarrow$ `AC Portable`.
5. **No Default Subcategories**:
   - Per decision, units linked directly to root categories will be evaluated on a case-by-case basis during asset data cleaning.
6. **Output Directories**:
   - `data/smart/staging/subcategory/`: Staging dataset with legacy `ams_code`, `parent_ams_code`, and 1-to-1 lookup table (`subcategory_ams_mapping.csv`).
   - `data/smart/final/subcategory/`: Production CSV (strictly the 8 target columns) and executable SQL script (`subcategories.sql`).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------
# Dynamic Path Configurations
# Robustly resolves data directory regardless of whether
# current working directory is /home/jovyan, /home/jovyan/work,
# or the repository root.
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path("data"),                                  # Running from workspace root or /home/jovyan
        Path("../data"),                                 # Running from notebooks/ or /home/jovyan/work
        Path("/home/jovyan/data"),                       # Absolute path inside jupyter container
        Path("/home/ran/Devs/jupyter-ams-smart/data"),   # Absolute path on host
    ]
    for p in candidates:
        if p.exists() and (p / "ams").exists():
            return p.resolve()
    return Path("../data").resolve()

DATA_DIR = get_data_dir()
RAW_DATA_PATH = DATA_DIR / "ams" / "tb_assets_category_202609291319.csv"
CAT_MAPPING_PATH = DATA_DIR / "smart" / "staging" / "category" / "category_ams_mapping.csv"
CAT_FINAL_PATH = DATA_DIR / "smart" / "final" / "category" / "categories.csv"

STAGING_DIR = DATA_DIR / "smart" / "staging" / "subcategory"
FINAL_DIR = DATA_DIR / "smart" / "final" / "subcategory"

# Ensure target directories exist
STAGING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Paths initialized (CWD: {Path.cwd()}):")
print(f" - Data Root:    {DATA_DIR}")
print(f" - Raw Data:     {RAW_DATA_PATH}")
print(f" - Cat Mapping:  {CAT_MAPPING_PATH}")
print(f" - Staging Dir:  {STAGING_DIR}")
print(f" - Final Dir:    {FINAL_DIR}")


Paths initialized (CWD: /home/jovyan/work):
 - Data Root:    /home/jovyan/data
 - Raw Data:     /home/jovyan/data/ams/tb_assets_category_202609291319.csv
 - Cat Mapping:  /home/jovyan/data/smart/staging/category/category_ams_mapping.csv
 - Staging Dir:  /home/jovyan/data/smart/staging/subcategory
 - Final Dir:    /home/jovyan/data/smart/final/subcategory


In [2]:
# Load raw AMS category dataset
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    na_values=["NULL", "null", "NaN", "nan", ""],
    keep_default_na=True,
    dtype=str
)

# Strip whitespace from string columns
for col in df_raw.select_dtypes(include=["object"]).columns:
    df_raw[col] = df_raw[col].str.strip()

# Load Category Relational Mapping Table generated by category.ipynb
df_cat_map = pd.read_csv(CAT_MAPPING_PATH, dtype=str)
df_cat_map["ams_code"] = df_cat_map["ams_code"].astype(int)
df_cat_map["category_id"] = df_cat_map["category_id"].astype(int)

print(f"Loaded {len(df_raw)} total rows from AMS category dataset.")
print(f"Loaded {len(df_cat_map)} category relational mapping records.")
display(df_cat_map.head(12))


# Load Final Category table for valid FK reference
df_cat_final = pd.read_csv(CAT_FINAL_PATH)


Loaded 128 total rows from AMS category dataset.
Loaded 15 category relational mapping records.


,ams_code,ams_id,category_id,new_code,new_name
0,100000,51,1,FUR,Furniture
1,110000,52,2,ACC,Aksesoris
2,120000,58,3,COMP,Computer
3,130000,59,4,MON,Monitor
4,140000,60,5,PERP,Peripheral
5,150000,61,6,PRNT,Printer
6,160000,62,7,SOFT,Software
7,170000,112,8,PROP,Properti & Gedung
8,220000,1142,8,PROP,Properti & Gedung
9,180000,113,9,MES,Mesin


## 3. Exploratory Data Analysis & Anomaly Detection

We filter for subcategories (`control == "T"`) and link them to their parent categories.
We also analyze potential code collisions (such as `ACC-WH`) and duplicate subcategories across categories.


In [3]:
# Filter for all subcategories (control == 'T')
df_subs = df_raw[df_raw["control"] == "T"].copy()
df_subs["ams_code_int"] = df_subs["code"].astype(int)
df_subs["cat_ams_code"] = (df_subs["ams_code_int"] // 10000) * 10000

# Merge with parent category mapping
df_subs = df_subs.merge(
    df_cat_map[["ams_code", "category_id", "new_code", "new_name"]],
    left_on="cat_ams_code",
    right_on="ams_code",
    how="left"
)

# Verify zero unmapped parents
unmapped = df_subs[df_subs["category_id"].isna()]
assert unmapped.empty, f"Error: Found unmapped subcategories: {unmapped['code'].tolist()}"

# Reassign telecommunication subcategories to category_id 13 (TLKM - Telekomunikasi)
TELECOM_RAW_IDS = [102, 80, 1195, 1210, 1211]
telecom_mask = df_subs["id"].astype(int).isin(TELECOM_RAW_IDS)
df_subs.loc[telecom_mask, "category_id"] = 13
df_subs.loc[telecom_mask, "new_code"] = "TLKM"
df_subs.loc[telecom_mask, "new_name"] = "Telekomunikasi"

# Reassign Water Heater (id: 1191) to category_id 9 (MES - Mesin)
wh_mask = df_subs["id"].astype(int) == 1191
df_subs.loc[wh_mask, "category_id"] = 9
df_subs.loc[wh_mask, "new_code"] = "MES"
df_subs.loc[wh_mask, "new_name"] = "Mesin"


print(f"Total subcategories in AMS (control == 'T'): {len(df_subs)} rows")
print(f"Parent category distribution across subcategories:")
print(df_subs["new_name"].value_counts())

# Inspect code collision for WH under ACC
wh_rows = df_subs[(df_subs["new_code"] == "ACC") & (df_subs["numbering_code"] == "WH")]
print("\nDetected numbering_code collision under ACC:")
display(wh_rows[["id", "code", "numbering_code", "category", "description"]])

# Inspect duplicate LIFT rows
lift_rows = df_subs[df_subs["category"].str.upper() == "LIFT"]
print("\nDetected duplicate LIFT rows across categories:")
display(lift_rows[["id", "code", "parent", "numbering_code", "category", "new_code", "new_name"]])

# Inspect Kendaraan / Mobil rows
kend_rows = df_subs[df_subs["new_code"] == "KEND"]
print("\nKendaraan / Mobil rows:")
display(kend_rows[["id", "code", "parent", "numbering_code", "category", "description"]])


Total subcategories in AMS (control == 'T'): 113 rows
Parent category distribution across subcategories:
new_name
Aksesoris            40
Elektronik           19
Peripheral           14
Furniture             8
Printer               6
Telekomunikasi        5
Pendingin Ruangan     5
Mesin                 5
Computer              4
Monitor               2
Software              2
Kendaraan             2
Properti & Gedung     1
Name: count, dtype: int64

Detected numbering_code collision under ACC:


,id,code,numbering_code,category,description
33,100,110002,WH,WHITEBOARD,Ada di ruang direksi dan sekar jagad



Detected duplicate LIFT rows across categories:


,id,code,parent,numbering_code,category,new_code,new_name
64,1177,180005,NaN,LIF,LIFT,MES,Mesin
101,1221,240017,ELK,LT,LIFT,ELEK,Elektronik



Kendaraan / Mobil rows:


,id,code,parent,numbering_code,category,description
94,1214,200001,NaN,KEN,KENDARAAN,Operasional RE
106,1226,200002,KEN,MO,MOBIL,NaN


## 4. Subcategory Cleaning, Deduplication & Transformation Pipeline

### Business Rules & Custom Transformations:
1. **Water Heater Reclassification & Code Standardization**:
   - Water Heater (`id: 1191`) $\rightarrow$ Reclassified under **Mesin (`MES`)** as `child_code = "WTH"` (**`MES-WTH`**).
   - Whiteboard (`id: 100`) $\rightarrow$ Remains in `Aksesoris` (`ACC`) as `child_code = "WB"` (**`ACC-WB`**).
2. **Duplicate LIFT Consolidation**:
   - Row `1221` (code `240017` in Elektronik) is merged into Row `1177` (code `180005` in Mesin) as **`MES-LIF`**.
   - Both AMS codes (`180005` and `240017`) map to `MES-LIF` in the lookup table.
3. **Kendaraan / Mobil Consolidation**:
   - Row `1214` (code `200001` *Operasional RE*) and Row `1226` (code `200002` *Mobil*) are consolidated into **`KEND-MO`** (`Mobil`).
   - Both AMS codes (`200001` and `200002`) map to `KEND-MO` in the lookup table.
4. **Name Normalization & Typos**:
   - Names standardized to **Title Case**.
   - Spelling corrected: `AC FORTABLE` $\rightarrow$ `Ac Portable`, `Celling Speaker` $\rightarrow$ `Ceiling Speaker`.
5. **Fixed Assets Setting**:
   - `is_consumable` set to `False` (`0`).

6. **Sentence Case Description Formatting**:
   - Descriptions standardized to **Sentence case** while preserving acronyms and proper names (e.g., `AC conditioner`, `Merk Seiko`, `Ada di ruang direksi dan Sekar Jagad`, `Operasional RE`).


In [4]:
# Helper to format names to Title Case while keeping acronyms clean
SPECIAL_NAME_CASING = {
    "AC Split": "AC Split",
    "AC AHU": "AC AHU",
    "AC Standing": "AC Standing",
    "AC CENTRAL": "AC Central",
    "AC FORTABLE": "AC Portable",
    "Celling Speaker": "Ceiling Speaker",
    "HT": "HT",
    "FAX": "Fax",
    "LCD": "LCD",
    "CRT": "Tabung",
    "UPS": "UPS",
    "VGA": "VGA",
    "TV": "TV",
    "CCTV": "CCTV",
    "GPS": "GPS",
    "ENGINEERING SOFT": "Engineering Software",
    "GENERAL SOFT": "General Software",
    "APAR": "Apar",
    "KOTAK P3K": "Kotak P3k",
    "WHITEBOARD": "Whiteboard",
    "WATER HEATER": "Water Heater",
    "LIFT": "Lift",
    "MOBIL": "Mobil",
    "KENDARAAN": "Mobil",
    "FOTO-FOTO": "Foto-Foto",
    "GLASSBOARD": "Glassboard",
    "ID CARD": "Id Card",
    "SMART EX03": "Smartphone",
    "LCEX S03": "Smartphone",
    "STORAGE TOOL BOX": "Storage Tool Box",
    "PAPPER CUTTER": "Paper Cutter",
    "PAPPER SHREDDER": "Paper Shredder",
    "FINGERPRINT READER": "Fingerprint Reader",
    "SMOKE DETECTOR": "Smoke Detector",
    "HEAT DETECTOR": "Heat Detector",
    "MULTIGAS DETECTOR": "Multigas Detector",
    "Figging Vume Control": "Fogging Volume Control",
    "Figging Control Toa": "Fogging Control Toa"
}

DESCRIPTION_OVERRIDES = {
    "Kursi Kerja": "Kursi kerja",
    "WArna Abu-abu dan cream": "Warna abu-abu dan cream",
    "Warna cream dan abu-abu": "Warna cream dan abu-abu",
    "Lemari dari thn 2011 - 2015": "Lemari dari thn 2011 - 2015",
    "Apar": "Apar",
    "Ada di ruang direksi dan sekar jagad": "Ada di ruang direksi dan Sekar Jagad",
    "Foto Departemen dan foto direksi": "Foto departemen dan foto direksi",
    "Merk Panasonic": "Merk Panasonic",
    "Merk SEIKO": "Merk Seiko",
    "Di ruang para Kadep dan PM": "Di ruang para Kadep dan PM",
    "Berbagai model dan bentuk": "Berbagai model dan bentuk",
    "Setiap Ruangan": "Setiap ruangan",
    "Setiap Ruangan ": "Setiap ruangan",
    "Merk Glant": "Merk Glant",
    "Merk Glant ": "Merk Glant",
    "Kotak": "Kotak",
    "Merk LG, SHARP dan Daikin": "Merk LG, Sharp dan Daikin",
    "Ac Condisioner": "AC conditioner",
    "DI ruang sekar jagad dan tiga negeri": "Di ruang Sekar Jagad dan Tiga Negeri",
    "Kursi Plastik Warna Cokelat": "Kursi plastik warna cokelat",
    "Cermin Cembung 45cm": "Cermin cembung 45cm",
    "Operasional RE": "Operasional RE",
    "Alat proyek": "Alat proyek"
}

def clean_description(raw_desc):
    if pd.isna(raw_desc) or str(raw_desc).strip() in ["", "NULL", "null"]:
        return None
    s = str(raw_desc).strip()
    if s in DESCRIPTION_OVERRIDES:
        return DESCRIPTION_OVERRIDES[s]
    return s[:1].upper() + s[1:].lower()

def clean_name(raw_name):
    if raw_name in SPECIAL_NAME_CASING:
        return SPECIAL_NAME_CASING[raw_name]
    # Clean words and Title Case
    return str(raw_name).strip().title()

# Code overrides to prevent collisions or improve clarity
CHILD_CODE_OVERRIDES = {
    100: "WB",      # Whiteboard (ACC-WB)
    1191: "WTH",    # Water Heater (MES-WTH)
    1168: "ACP",    # AC Portable (PDGR-ACP)
    121: "ACS",     # AC Standing (PDGR-ACS)
    119: "ACSP",    # AC Split (PDGR-ACSP)
    1175: "ACN",    # AC Central (PDGR-ACN)
    120: "AHU",     # AC AHU (PDGR-AHU)
    1226: "MO",     # Mobil (KEND-MO)
    102: "TL",      # Telepon (TLKM-TL)
    1210: "SP",     # Smartphone (TLKM-SP)
    1195: "HT",     # HT (TLKM-HT)
    80: "FAX",      # Fax (TLKM-FAX)
}

# Subcategories to be consolidated into primary rows:
# - Row 1221 (LIFT in Elektronik) merges into Row 1177 (LIFT in Mesin)
# - Row 1214 (KENDARAAN in Kendaraan) merges into Row 1226 (MOBIL in Kendaraan)
# - Row 1211 (LCEX S03 in Aksesoris) merges into Row 1210 (SMART EX03 / Smartphone in Telekomunikasi)
MERGE_TARGET_MAP = {
    1221: 1177,     # 240017 (LIFT) -> 180005 (LIFT in Mesin)
    1214: 1226,     # 200001 (KENDARAAN Operasional) -> 200002 (MOBIL in Kendaraan)
    1211: 1210      # 110030 (LCEX S03) -> 240015 (SMART EX03 / Smartphone)
}

# Filter out secondary merged rows for the master subcategory list
primary_subs = df_subs[~df_subs["id"].astype(int).isin(MERGE_TARGET_MAP.keys())].copy()

# Order primary subcategories:
# 1. Furniture (category_id: 1) -> IDs 1..8
# 2. Aksesoris before Telepon (category_id: 2, ams_code < 110004) -> IDs 9..11
# 3. Telepon (id: 102, re-assigned to TLKM category_id: 13) strictly pinned at ID 12
# 4. Remaining Aksesoris (category_id: 2, ams_code > 110004) -> IDs 13..49
# 5. Master categories 3..12 sorted by category_id, code -> IDs 51..107
# 6. Remaining Telekomunikasi subcategories (FAX, HT, Smartphone) -> IDs 108..110
non_telecom = primary_subs[primary_subs["category_id"] != 13].copy()
telecom = primary_subs[primary_subs["category_id"] == 13].copy()

part_fur = non_telecom[non_telecom["category_id"] == 1].sort_values(by="code")
part_acc = non_telecom[non_telecom["category_id"] == 2].sort_values(by="code")
part_acc_before = part_acc[part_acc["ams_code_int"] < 110004]
part_acc_after = part_acc[part_acc["ams_code_int"] > 110004]

telepon_row = telecom[telecom["id"].astype(int) == 102]
other_telecom = telecom[telecom["id"].astype(int) != 102].sort_values(by="code")
other_cats = non_telecom[non_telecom["category_id"] > 2].sort_values(by=["category_id", "code"])

primary_subs = pd.concat([
    part_fur,
    part_acc_before,
    telepon_row,
    part_acc_after,
    other_cats,
    other_telecom
]).reset_index(drop=True)

# Build Master Staging & Final records
staging_records = []
final_records = []
mapping_records = []

for idx, r in primary_subs.iterrows():
    sub_id = idx + 1
    raw_id = int(r["id"])
    parent_cat_id = int(r["category_id"])
    parent_cat_code = r["new_code"]
    parent_ams_code = int(r["cat_ams_code"])
    
    # Child code resolution
    child_code = CHILD_CODE_OVERRIDES.get(raw_id, str(r["numbering_code"]).strip())
    sub_code = f"{parent_cat_code}-{child_code}"
    
    # Name & description
    sub_name = clean_name(r["category"])
    sub_desc = clean_description(r["description"])
    
    # Legacy codes mapping
    ams_code = int(r["code"])
    ams_codes = [ams_code]
    ams_ids = [raw_id]
    
    # Check if any merged rows point to this primary row
    for secondary_id, target_id in MERGE_TARGET_MAP.items():
        if target_id == raw_id:
            sec_row = df_subs[df_subs["id"].astype(int) == secondary_id].iloc[0]
            sec_ams_code = int(sec_row["code"])
            ams_codes.append(sec_ams_code)
            ams_ids.append(secondary_id)
    
    # Timestamps
    created_at = r["create_date"] if pd.notna(r["create_date"]) else datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    updated_at = created_at
    
    # Staging record
    staging_records.append({
        "id": sub_id,
        "code": sub_code,
        "name": sub_name,
        "description": sub_desc,
        "is_consumable": False,
        "category_id": parent_cat_id,
        "category_code": parent_cat_code,
        "parent_ams_code": parent_ams_code,
        "ams_codes": str(ams_codes),
        "ams_ids": str(ams_ids),
        "created_at": created_at,
        "updated_at": updated_at
    })
    
    # Final production record (strictly 8 target columns)
    final_records.append({
        "id": sub_id,
        "code": sub_code,
        "name": sub_name,
        "description": sub_desc,
        "is_consumable": False, # All are assets, not consumable
        "category_id": parent_cat_id,
        "created_at": created_at,
        "updated_at": updated_at
    })
    
    # 1-to-1 Mapping records for every legacy code
    for ac, ai in zip(ams_codes, ams_ids):
        mapping_records.append({
            "ams_code": ac,
            "ams_id": ai,
            "subcategory_id": sub_id,
            "subcategory_code": sub_code,
            "subcategory_name": sub_name,
            "category_id": parent_cat_id,
            "category_code": parent_cat_code
        })

df_sub_staging = pd.DataFrame(staging_records)
df_sub_final = pd.DataFrame(final_records)
df_sub_mapping = pd.DataFrame(mapping_records)

print(f"Pipeline completed successfully:")
print(f" - Primary Subcategories: {len(df_sub_final)} rows (consolidated from 113 AMS rows)")
print(f" - Legacy 1-to-1 Mapping Entries: {len(df_sub_mapping)} rows (covers 100% of legacy subcategory codes)")

print(f"\nPreview of Final Subcategories (first 12 rows):")
display(df_sub_final.head(12))


Pipeline completed successfully:
 - Primary Subcategories: 110 rows (consolidated from 113 AMS rows)
 - Legacy 1-to-1 Mapping Entries: 113 rows (covers 100% of legacy subcategory codes)

Preview of Final Subcategories (first 12 rows):


,id,code,name,description,is_consumable,category_id,created_at,updated_at
0,1,FUR-KK,Kursi Kerja,Kursi kerja,False,1,2026-04-08 08:24:28.000,2026-04-08 08:24:28.000
1,2,FUR-MK,Meja Kerja,Warna abu-abu dan cream,False,1,2026-04-08 08:24:37.000,2026-04-08 08:24:37.000
2,3,FUR-LC,Laci Kerja,Warna cream dan abu-abu,False,1,2026-04-08 08:24:44.000,2026-04-08 08:24:44.000
3,4,FUR-LM,Lemari,Lemari dari thn 2011 - 2015,False,1,2026-04-08 08:24:57.000,2026-04-08 08:24:57.000
4,5,FUR-KP,Kursi Plastik,Kursi plastik warna cokelat,False,1,2026-04-08 08:25:03.000,2026-04-08 08:25:03.000
5,6,FUR-MB,Meja Bundar,None,False,1,2026-04-08 08:25:10.000,2026-04-08 08:25:10.000
6,7,FUR-KT,Kursi Tamu,None,False,1,2026-04-08 08:25:40.000,2026-04-08 08:25:40.000
7,8,FUR-MT,Meja Tamu,None,False,1,2026-04-08 08:25:35.000,2026-04-08 08:25:35.000
8,9,ACC-AP,Apar,Apar,False,2,2026-04-08 08:25:46.000,2026-04-08 08:25:46.000
9,10,ACC-WB,Whiteboard,Ada di ruang direksi dan Sekar Jagad,False,2,2026-04-08 08:25:57.000,2026-04-08 08:25:57.000


## 5. Data Validation & Quality Checks
We run assertions and integrity checks before exporting to ensure:
1. `code` values are strictly unique and $\le 9$ characters (`VARCHAR(9)`).
2. `category_id` values reference valid categories.
3. No unexpected null values exist in required fields.
4. All 113 AMS subcategory rows are accounted for in the lookup mapping table.


In [5]:
# 1. Check code length constraint (VARCHAR(9))
max_code_len = df_sub_final["code"].str.len().max()
assert max_code_len <= 9, f"Error: Maximum code length exceeds 9 characters ({max_code_len})"

# 2. Check code uniqueness
duplicate_codes = df_sub_final[df_sub_final["code"].duplicated(keep=False)]
assert duplicate_codes.empty, f"Error: Duplicate subcategory codes detected: {duplicate_codes['code'].tolist()}"

# 3. Check foreign key validity (category_id must exist in category master)
valid_category_ids = set(df_cat_final["id"].astype(int))
invalid_fk = df_sub_final[~df_sub_final["category_id"].isin(valid_category_ids)]
assert invalid_fk.empty, f"Error: Found invalid category_id values: {invalid_fk['category_id'].tolist()}"

# 4. Check for NULL values in required columns
required_cols = ["id", "code", "name", "is_consumable", "category_id", "created_at", "updated_at"]
null_counts = df_sub_final[required_cols].isna().sum()
assert null_counts.sum() == 0, f"Error: Found NULL values in required fields: {null_counts.to_dict()}"

# 5. Check sequential IDs and Telepon / Smartphone constraints
assert df_sub_final["id"].is_monotonic_increasing, "Error: IDs are not monotonic increasing"
assert len(df_sub_final) == 110, f"Error: Expected 110 subcategories, found {len(df_sub_final)}"

tel = df_sub_final[df_sub_final["id"] == 12].iloc[0]
assert tel["name"] == "Telepon" and tel["code"] == "TLKM-TL" and tel["category_id"] == 13, f"Telepon assertion failed: {tel.to_dict()}"

sp = df_sub_final[df_sub_final["name"] == "Smartphone"].iloc[0]
assert sp["id"] == 110 and sp["code"] == "TLKM-SP" and sp["category_id"] == 13, f"Smartphone assertion failed: {sp.to_dict()}"

# 6. Check that all 113 legacy subcategories are mapped
assert len(df_sub_mapping) == 113, f"Error: Expected 113 mapped legacy codes, found {len(df_sub_mapping)}"

print("ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY!")
print(f" - Total Production Subcategories: {len(df_sub_final)}")
print(f" - Max Code Length: {max_code_len} characters (limit: 9)")
print(f" - Unique Codes: {df_sub_final['code'].nunique()} / {len(df_sub_final)}")
print(f" - Total Legacy AMS Codes Mapped: {len(df_sub_mapping)} / 113")


ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY!
 - Total Production Subcategories: 110
 - Max Code Length: 9 characters (limit: 9)
 - Unique Codes: 110 / 110
 - Total Legacy AMS Codes Mapped: 113 / 113


## 6. Export Outputs

We export staging files and final production files in **two separate cells** as established:
1. **Cell 11**: Staging files (`data/smart/staging/subcategory/`)
   - `subcategories.csv`
   - `subcategory_ams_mapping.csv`
2. **Cell 12**: Final production files (`data/smart/final/subcategory/`)
   - `subcategories.csv` (Strict 8 target columns for MS SQL Server)
   - `subcategories.sql` (Executable MS SQL Server script with `IDENTITY_INSERT`)


In [6]:
# ---------------------------------------------------------
# Export Staging Outputs to data/smart/staging/subcategory/
# ---------------------------------------------------------
staging_sub_path = STAGING_DIR / "subcategories.csv"
staging_map_path = STAGING_DIR / "subcategory_ams_mapping.csv"

# 1. Staging subcategories with legacy codes and parent info
df_sub_staging.to_csv(staging_sub_path, index=False)

# 2. 1-to-1 Relational lookup mapping table
df_sub_mapping.to_csv(staging_map_path, index=False)

print(f"Staging files successfully exported to {STAGING_DIR.resolve()}:")
print(f" 1. {staging_sub_path.name} ({len(df_sub_staging)} rows)")
print(f" 2. {staging_map_path.name} ({len(df_sub_mapping)} rows)")


Staging files successfully exported to /home/jovyan/data/smart/staging/subcategory:
 1. subcategories.csv (110 rows)
 2. subcategory_ams_mapping.csv (113 rows)


In [7]:
# ---------------------------------------------------------
# Export Production Outputs to data/smart/final/subcategory/
# ---------------------------------------------------------
final_csv_path = FINAL_DIR / "subcategories.csv"
final_sql_path = FINAL_DIR / "subcategories.sql"

# 1. Target CSV for SQL Server Import (Strict 8 Columns)
df_sub_final.to_csv(final_csv_path, index=False)

# 2. Generate SQL Server INSERT Script
sql_statements = [
    "-- ========================================================",
    "-- Master Subcategories Data Migration Script for MS SQL Server",
    f"-- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "-- Target Table: subcategories",
    "-- ========================================================",
    "SET IDENTITY_INSERT [subcategories] ON;",
    "GO",
    ""
]

for _, row in df_sub_final.iterrows():
    s_id = row["id"]
    s_code = str(row["code"]).replace("'", "''")
    s_name = str(row["name"]).replace("'", "''")
    if pd.notna(row["description"]):
        escaped_desc = str(row["description"]).replace("'", "''")
        s_desc = f"N'{escaped_desc}'"
    else:
        s_desc = "NULL"
    s_consumable = 0
    s_cat_id = row["category_id"]
    s_created = row["created_at"]
    s_updated = row["updated_at"]
    
    sql = (
        f"INSERT INTO [subcategories] ([id], [code], [name], [description], [is_consumable], [category_id], [created_at], [updated_at]) "
        f"VALUES ({s_id}, '{s_code}', N'{s_name}', {s_desc}, {s_consumable}, {s_cat_id}, '{s_created}', '{s_updated}');"
    )
    sql_statements.append(sql)

sql_statements.extend([
    "",
    "GO",
    "SET IDENTITY_INSERT [subcategories] OFF;",
    "GO"
])

with open(final_sql_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements))

print(f"Production files successfully exported to {FINAL_DIR.resolve()}:")
print(f" 1. {final_csv_path.name} ({len(df_sub_final)} rows)")
print(f" 2. {final_sql_path.name} ({len(df_sub_final)} SQL insert statements)")
print("\n--- SQL Script Preview (First 5 statements) ---")
print("\n".join(sql_statements[:15]))


Production files successfully exported to /home/jovyan/data/smart/final/subcategory:
 1. subcategories.csv (110 rows)
 2. subcategories.sql (110 SQL insert statements)

--- SQL Script Preview (First 5 statements) ---
-- ========================================================
-- Master Subcategories Data Migration Script for MS SQL Server
-- Generated: 2026-10-01 07:52:45
-- Target Table: subcategories
-- ========================================================
SET IDENTITY_INSERT [subcategories] ON;
GO

INSERT INTO [subcategories] ([id], [code], [name], [description], [is_consumable], [category_id], [created_at], [updated_at]) VALUES (1, 'FUR-KK', N'Kursi Kerja', N'Kursi kerja', 0, 1, '2026-04-08 08:24:28.000', '2026-04-08 08:24:28.000');
INSERT INTO [subcategories] ([id], [code], [name], [description], [is_consumable], [category_id], [created_at], [updated_at]) VALUES (2, 'FUR-MK', N'Meja Kerja', N'Warna abu-abu dan cream', 0, 1, '2026-04-08 08:24:37.000', '2026-04-08 08:24:37.000');